# Assignment 4: Clustering, Web Search, and PageRank

**Name:** Chirag Phor  
**Roll Number:** m25de1021
**GitHub Link:** https://github.com/cph0r/ass-4

This notebook contains the solutions for the assignment.

In [1]:
# Colab Setup: Run this cell to fetch datasets if in Google Colab
import os
if not os.path.exists("Assignment 4- datasets"):
    !git clone https://github.com/cph0r/ass-4.git
    %cd ass-4


## Part 1: Clustering
Implementation of Farthest First Traversal (k-center) and k-means++ algorithms on the UCI Spam dataset.

In [2]:
import numpy as np
import time

def readVectorsSeq(filename):
    data = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                features = list(map(float, line.split(',')))
                data.append(features)
    return np.array(data)

def kcenter(P, k):
    P_arr = np.array(P)
    n = P_arr.shape[0]
    centers_idx = [0]
    min_dists = np.sum((P_arr - P_arr[0])**2, axis=1)
    
    for _ in range(1, k):
        next_center_idx = np.argmax(min_dists)
        centers_idx.append(next_center_idx)
        dist_to_new = np.sum((P_arr - P_arr[next_center_idx])**2, axis=1)
        min_dists = np.minimum(min_dists, dist_to_new)
        
    return P_arr[centers_idx]

def kmeansPP(P, k):
    P_arr = np.array(P)
    n = P_arr.shape[0]
    centers_idx = [np.random.randint(n)]
    min_dists = np.sum((P_arr - P_arr[centers_idx[0]])**2, axis=1)
    
    for _ in range(1, k):
        d_sum = np.sum(min_dists)
        if d_sum == 0:
            probs = np.ones(n) / n
        else:
            probs = min_dists / d_sum
            
        next_center = np.random.choice(n, p=probs)
        centers_idx.append(next_center)
        dist_to_new = np.sum((P_arr - P_arr[next_center])**2, axis=1)
        min_dists = np.minimum(min_dists, dist_to_new)
        
    return P_arr[centers_idx]

def kmeansObj(P, C):
    P_arr = np.array(P)
    C_arr = np.array(C)
    dists = np.linalg.norm(P_arr[:, np.newaxis] - C_arr, axis=2)**2
    min_dists = np.min(dists, axis=1)
    return np.mean(min_dists)

In [3]:
# Main Execution for Part 1
P = readVectorsSeq('Assignment 4- datasets/Q1- UCI Spam clustering/spambase.data')
k = 10
k1 = 20 

# Step 1
start_time = time.time()
centers_kcenter = kcenter(P, k)
end_time = time.time()
print(f"Step 1 - kcenter execution time: {end_time - start_time:.4f} seconds")

# Step 2
centers_kmeanspp = kmeansPP(P, k)
obj_kmeanspp = kmeansObj(P, centers_kmeanspp)
print(f"Step 2 - kmeansPP objective: {obj_kmeanspp:.4f}")

# Step 3
X = kcenter(P, k1)
C = kmeansPP(X, k)
obj_step3 = kmeansObj(P, C)
print(f"Step 3 - Advanced Strategy objective (kcenter with k1={k1} -> kmeansPP with k={k}): {obj_step3:.4f}")

Step 1 - kcenter execution time: 0.0038 seconds
Step 2 - kmeansPP objective: 29144.3638
Step 3 - Advanced Strategy objective (kcenter with k1=20 -> kmeansPP with k=10): 1022766.5385


## Part 2: Web Search
Inverted Index + TF-IDF implementation using Object-Oriented principles.

In [4]:
import os
import math

class MySet:
    def __init__(self):
        self.elements = []
        
    def addElement(self, el):
        if el not in self.elements:
            self.elements.append(el)
            
    def union(self, otherSet):
        res = MySet()
        for x in self.elements:
            res.addElement(x)
        for x in otherSet.elements:
            res.addElement(x)
        return res
        
    def intersection(self, otherSet):
        res = MySet()
        for x in self.elements:
            if x in otherSet.elements:
                res.addElement(x)
        return res

class Position:
    def __init__(self, page, wordIndex):
        self.page = page 
        self.wordIndex = wordIndex

class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = []
        
    def addPosition(self, position):
        self.positions.append(position)
        
    def getTermFrequency(self, page_name):
        return sum(1 for p in self.positions if p.page.page_name == page_name)

class PageIndex:
    def __init__(self):
        self.word_entries = []
        
    def addPositionForWord(self, str_word, p):
        for we in self.word_entries:
            if we.word == str_word:
                we.addPosition(p)
                return
        we = WordEntry(str_word)
        we.addPosition(p)
        self.word_entries.append(we)

def preprocess_word(w):
    w = w.lower()
    punctuations = "{}[]<>()=.,;'\"?#!-:"
    for p in punctuations:
        w = w.replace(p, ' ')
        
    w = w.split()[0] if w.strip() else ""
    
    if w == "stacks": w = "stack"
    elif w == "structures": w = "structure"
    elif w == "applications": w = "application"
    return w

STOP_WORDS = ["a", "an", "the", "they", "these", "this", "for", "is", "are", "was", "of", "or", "and", "does", "will", "whose"]

class PageEntry:
    def __init__(self, page_name, directory='Assignment 4- datasets/Q2- webSearch/webpages'):
        self.page_name = page_name
        self.page_index = PageIndex()
        self.total_words = 0
        
        filepath = os.path.join(directory, page_name)
        if not os.path.exists(filepath):
            return
            
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
            
        words = content.split()
        idx = 1
        for w in words:
            w_proc = preprocess_word(w)
            if not w_proc or w_proc in STOP_WORDS:
                idx += 1
                continue
                
            self.total_words += 1
            p = Position(self, idx)
            self.page_index.addPositionForWord(w_proc, p)
            idx += 1

class MyHashTable:
    def __init__(self):
        self.table = {}
        
    def addWordEntry(self, w_entry):
        self.table[w_entry.word] = w_entry
        
    def getWordEntry(self, word):
        return self.table.get(word, None)

class InvertedPageIndex:
    def __init__(self):
        self.hash_table = MyHashTable()
        self.pages = []
        
    def addPage(self, page):
        self.pages.append(page)
        for w_entry in page.page_index.word_entries:
            existing_we = self.hash_table.getWordEntry(w_entry.word)
            if existing_we is None:
                new_we = WordEntry(w_entry.word)
                for pos in w_entry.positions:
                    new_we.addPosition(pos)
                self.hash_table.addWordEntry(new_we)
            else:
                for pos in w_entry.positions:
                    existing_we.addPosition(pos)
                    
    def getPagesWhichContainWord(self, str_word):
        w = preprocess_word(str_word)
        we = self.hash_table.getWordEntry(w)
        if we is None:
            return MySet()
        ans = MySet()
        for p in we.positions:
            ans.addElement(p.page)
        return ans
        
    def getRelevance(self, str_word, page_name):
        w_proc = preprocess_word(str_word)
        we = self.hash_table.getWordEntry(w_proc)
        if not we: return 0.0
        
        p_entry = next((p for p in self.pages if p.page_name == page_name), None)
        if not p_entry or p_entry.total_words == 0: return 0.0
        
        tf = we.getTermFrequency(page_name) / p_entry.total_words
        N = len(self.pages)
        pages_containing_w = len(set(p.page.page_name for p in we.positions))
        idf = math.log10(N / pages_containing_w) if pages_containing_w > 0 else 0
        return tf * idf

class SearchEngine:
    def __init__(self):
        self.inv_idx = InvertedPageIndex()
        
    def performAction(self, actionMessage):
        parts = actionMessage.strip().split()
        if not parts:
            return
        cmd = parts[0]
        if cmd == "addPage":
            page_name = parts[1]
            p = PageEntry(page_name)
            self.inv_idx.addPage(p)
        elif cmd == "queryFindPagesWhichContainWord":
            w = parts[1]
            pages_set = self.inv_idx.getPagesWhichContainWord(w)
            if not pages_set.elements:
                print(f"No webpage contains word {w}")
            else:
                names = [p.page_name for p in pages_set.elements]
                print(", ".join(sorted(names)))
        elif cmd == "queryFindPositionsOfWordInAPage":
            w = parts[1]
            page_name = parts[2]
            w_proc = preprocess_word(w)
            we = self.inv_idx.hash_table.getWordEntry(w_proc)
            if we is None:
                print(f"Webpage {page_name} does not contain word {w}")
                return
            
            positions = [str(p.wordIndex) for p in we.positions if p.page.page_name == page_name]
            if not positions:
                print(f"Webpage {page_name} does not contain word {w}")
            else:
                print(", ".join(positions))


In [5]:
# Main Execution for Part 2
se = SearchEngine()
try:
    print("Running Search Queries (matching actions.txt format):")
    with open('Assignment 4- datasets/Q2- webSearch/actions.txt', 'r') as f:
        for line in f:
            se.performAction(line)
except Exception as e:
    print("Error retrieving actions.txt:", e)

Running Search Queries (matching actions.txt format):
No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


## Part 3: PageRank on Spark
Iteration-based approach utilizing PySpark.

In [6]:
from pyspark.sql import SparkSession

def run_pagerank(file_path):
    spark = SparkSession.builder.appName("PageRank").getOrCreate()
    sc = spark.sparkContext
    
    try:
        lines = sc.textFile(file_path)
    except Exception:
        print(f"File {file_path} not found.")
        return
        
    if lines.isEmpty():
        print(f"File {file_path} is empty.")
        return
        
    edges = lines.map(lambda x: tuple(x.split())).filter(lambda x: len(x) == 2)
    edges = edges.distinct() 
    
    links = edges.groupByKey().mapValues(list).cache()
    
    nodes = edges.keys().union(edges.values()).distinct()
    n = nodes.count()
    
    ranks = nodes.map(lambda node: (node, 1.0 / n))
    
    beta = 0.8
    
    for _ in range(40):
        contribs = links.join(ranks).flatMap(
            lambda url_urls_rank: [(dest, url_urls_rank[1][1] / len(url_urls_rank[1][0])) for dest in url_urls_rank[1][0]]
        )
        
        sums = contribs.reduceByKey(lambda x, y: x + y)
        ranks = nodes.map(lambda x: (x, 0.0)).leftOuterJoin(sums).mapValues(lambda x: (1.0 - beta) / n + beta * (x[1] or 0.0))
        ranks.cache()
        ranks.count()  # Breaks lineage and stops executor timeouts!

    sorted_ranks = ranks.sortBy(lambda x: x[1], ascending=False).collect()
    
    print(f"--- PageRank Results for {file_path} ---")
    print("Top 5 nodes:")
    for node, rank in sorted_ranks[:5]:
        print(f"Node {node}: {rank}")
        
    print("\nBottom 5 nodes:")
    for node, rank in sorted_ranks[-5:]:
        print(f"Node {node}: {rank}")



In [7]:
# Main Execution for Part 3
run_pagerank('Assignment 4- datasets/small.txt')
run_pagerank('Assignment 4- datasets/whole.txt')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 14:10:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


--- PageRank Results for Assignment 4- datasets/small.txt ---
Top 5 nodes:
Node 53: 0.03573120223267161
Node 14: 0.03417090697259136
Node 40: 0.0336300871897439
Node 1: 0.030005979479788617
Node 27: 0.029720144201405393

Bottom 5 nodes:
Node 89: 0.003922466019802268
Node 37: 0.0038082042916114515
Node 81: 0.0036953517493609916
Node 59: 0.0036698606601272845
Node 85: 0.003409694077402821


--- PageRank Results for Assignment 4- datasets/whole.txt ---
Top 5 nodes:
Node 263: 0.002020291181518219
Node 537: 0.0019433415714531497
Node 965: 0.0019254478071662634
Node 243: 0.001852634016241731
Node 285: 0.0018273721700645144

Bottom 5 nodes:
Node 408: 0.00038779848719291705
Node 424: 0.00035481538649301443
Node 62: 0.00035314810510596274
Node 93: 0.00035135689375165774
Node 558: 0.0003286018525215297
